# 🍵 Fine-Tuner Lab — Green Neko Gen Z Assistant

Fine-tunes **Qwen2.5-0.5B-Instruct** with **LoRA/PEFT** on 80 Green Neko Cafe Q&A examples.  
Produces `results.json` with before/after outputs and metrics for the webapp.

**Runtime:** GPU → T4  
**Estimated training time:** ~5 minutes on a free T4

---
### Steps
1. Install dependencies  
2. Load base model (4-bit quantized)  
3. Run baseline inference (BEFORE)  
4. Prepare dataset & tokenize  
5. Configure LoRA + train with SFTTrainer  
6. Run fine-tuned inference (AFTER)  
7. Compute metrics  
8. Export `results.json`  
9. (Optional) Push adapter to HuggingFace Hub

In [ ]:
# ── Cell A: Install dependencies (pinned for Colab T4 stability) ──────────────
!pip install -q \
    transformers==4.44.0 \
    peft==0.12.0 \
    trl==0.11.0 \
    bitsandbytes==0.43.3 \
    datasets==3.0.0 \
    rouge-score==0.1.2 \
    accelerate==0.34.0

print('✅ Dependencies installed')

In [ ]:
# ── Cell B: Imports & config ──────────────────────────────────────────────────
import json
import math
import re
import torch
from pathlib import Path

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
from rouge_score import rouge_scorer

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
MAX_SEQ_LEN = 512

SLANG = [
    'no cap', 'lowkey', 'bussin', 'fr fr', 'slay', 'hits different',
    'vibe', 'aesthetic', 'ngl', 'bestie', 'understood the assignment',
    "it's giving", 'slaps', 'rent free', 'main character', 'based',
    'ate that', 'not mid at all', 'W move', 'core',
]

TEST_QUESTIONS = [
    "What's the best boba at Green Neko?",
    "I'm vegetarian — what should I order at Green Neko?",
    "Is the Salmon Poke Bowl worth the price?",
    "What even is Green Neko? Like what kind of food is it?",
    "What's the difference between the Matcha Latte and Matcha Tea Boba?",
]

REFERENCE_ANSWERS = [
    "ok bestie no cap the Matcha Tea Boba (₹290) is lowkey the main character move here — hits different with the earthy matcha taste. but ngl if you want something bussin and unique try the Mango Matcha Green Tea (₹290) — it's giving two vibes in one fr fr. for classic energy go Brown Sugar Milk Boba (₹250) that one slaps.",
    "bestie Green Neko actually understood the assignment for vegetarians no cap! appetizers: Teriyaki Tofu (₹300), Tofu Katsu Finger (₹320), Mushroom Tempura (₹320), Teriyaki Paneer (₹350), Paneer Katsu Finger (₹390). mains: Tofu Poke Bowl (₹430) or Paneer Poke Bowl (₹470). for drinks literally everything fr fr — all the boba, matcha lattes, it's giving options slay.",
    "ok fr fr ₹780 sounds like a lot but ngl when you factor in the pan-seared salmon AND avocado together it's actually bussin value for what you get in Delhi. it's lowkey the most slay dish on the menu — main character energy. if you're treating yourself understood the assignment bestie.",
    "fr fr Green Neko is a Japanese-Hawaiian bistro with Taiwanese drinks — lowkey a fusion concept that understood the assignment no cap. think poke bowls, Japanese katsu, takoyaki, and then pair everything with matcha boba or Vietnamese coffee. the vibe is where cozy meets modern. bestie it hits different from your average Delhi restaurant slay.",
    "ok bestie fr fr — Matcha Latte (₹300) is smooth and creamy milk-based, no pearls. Matcha Tea Boba (₹290) has the chewy tapioca pearls served cold — hits different. ngl the Latte is ₹10 more but the Boba gives main character aesthetic energy. understood the assignment for matcha lovers no cap.",
]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Using device: {device}')
if device == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell C: Load tokenizer + base model (4-bit NF4) ───────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
base_model.config.use_cache = False

print('✅ Base model loaded')
total_params = sum(p.numel() for p in base_model.parameters())
print(f'   Parameters: {total_params / 1e6:.1f}M')

In [ ]:
# ── Cell D: Baseline inference (BEFORE fine-tuning) ───────────────────────────
def run_inference(model, questions, max_new_tokens=200):
    model.eval()
    outputs = []
    with torch.no_grad():
        for q in questions:
            prompt = f'### Instruction:\n{q}\n\n### Response:\n'
            inputs = tokenizer(prompt, return_tensors='pt').to(device)
            input_len = inputs['input_ids'].shape[1]
            gen_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=1.0,
                pad_token_id=tokenizer.pad_token_id,
            )
            new_tokens = gen_ids[0][input_len:]
            text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            outputs.append({'text': text, 'tokens': len(new_tokens)})
            print(f'  Q: {q[:60]}...')
            print(f'  A: {text[:100]}...')
            print()
    return outputs

print('🔍 Running BEFORE inference...')
before_outputs = run_inference(base_model, TEST_QUESTIONS)
print('✅ Baseline inference complete')

In [ ]:
# ── Cell E: Upload & prepare training data ────────────────────────────────────
# Option 1: Upload train.jsonl from your local machine
from google.colab import files
print('📁 Upload train.jsonl from data/ folder...')
uploaded = files.upload()  # Upload data/train.jsonl when prompted

# Parse uploaded file
filename = list(uploaded.keys())[0]
examples = [json.loads(line) for line in uploaded[filename].decode('utf-8').splitlines() if line.strip()]
print(f'✅ Loaded {len(examples)} training examples')

# Format into Alpaca prompt template
def format_example(ex):
    if ex.get('input', '').strip():
        prompt = f"### Instruction:\n{ex['instruction']}\n\n### Input:\n{ex['input']}\n\n### Response:\n{ex['output']}"
    else:
        prompt = f"### Instruction:\n{ex['instruction']}\n\n### Response:\n{ex['output']}"
    return {'text': prompt}

formatted = [format_example(ex) for ex in examples]
dataset = Dataset.from_list(formatted)
print(f'✅ Dataset ready: {len(dataset)} examples')
print('Sample:')
print(dataset[0]['text'][:300])

In [ ]:
# ── Cell F: Configure LoRA + prepare model ────────────────────────────────────
# IMPORTANT: prepare_model_for_kbit_training MUST be called before get_peft_model
base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()
print('✅ LoRA applied')

In [ ]:
# ── Cell G: Train with SFTTrainer ─────────────────────────────────────────────
# IMPORTANT: fp16=True, bf16=False for T4 GPU (T4 doesn't support native bf16 in optimizer)

training_args = SFTConfig(
    output_dir='./checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # effective batch = 16
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    fp16=True,   # ← T4 safe
    bf16=False,  # ← must be False on T4
    logging_steps=5,
    save_strategy='epoch',
    report_to='none',  # disable wandb
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field='text',
    optim='paged_adamw_32bit',
)

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=dataset,
    args=training_args,
    tokenizer=tokenizer,
)

print('🚀 Starting training...')
train_result = trainer.train()
print('✅ Training complete!')
print(f'   Train loss: {train_result.training_loss:.4f}')

In [ ]:
# ── Cell H: Extract training history ─────────────────────────────────────────
training_history = [
    {'step': entry['step'], 'loss': round(entry['loss'], 4)}
    for entry in trainer.state.log_history
    if 'loss' in entry and 'step' in entry
]
print(f'✅ Captured {len(training_history)} loss checkpoints')
for entry in training_history:
    print(f"  Step {entry['step']:3d}: loss = {entry['loss']}")

In [ ]:
# ── Cell I: Fine-tuned inference (AFTER) ──────────────────────────────────────
print('🔍 Running AFTER inference...')
after_outputs = run_inference(peft_model, TEST_QUESTIONS)
print('✅ Fine-tuned inference complete')

In [ ]:
# ── Cell J: Compute metrics ───────────────────────────────────────────────────

def count_slang(text):
    text_lower = text.lower()
    return sum(1 for s in SLANG if s in text_lower)

def slang_density(text):
    """Gen Z slang words per 100 tokens (approx — use whitespace split)"""
    token_count = max(len(text.split()), 1)
    return round(count_slang(text) / token_count * 100, 2)

def compute_perplexity(model, tokenizer, texts, max_length=256):
    """Manual perplexity — DO NOT use evaluate.load('perplexity') on T4 (OOM)."""
    model.eval()
    total_loss = 0.0
    count = 0
    with torch.no_grad():
        for text in texts:
            enc = tokenizer(
                text, return_tensors='pt',
                max_length=max_length, truncation=True
            ).to(device)
            labels = enc['input_ids'].clone()
            out = model(**enc, labels=labels)
            total_loss += out.loss.item()
            count += 1
    return round(math.exp(total_loss / count), 2)

# Use 10% of training data as validation for perplexity
val_texts = [ex['text'] for ex in formatted[:8]]  # first 8 examples

print('📊 Computing perplexity (before)...')
ppl_before = compute_perplexity(base_model, tokenizer, val_texts)
print(f'   Perplexity BEFORE: {ppl_before}')

print('📊 Computing perplexity (after)...')
ppl_after = compute_perplexity(peft_model, tokenizer, val_texts)
print(f'   Perplexity AFTER:  {ppl_after}')

# ROUGE-L
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def avg_rouge_l(outputs, references):
    scores = [scorer.score(ref, out['text'])['rougeL'].fmeasure
              for out, ref in zip(outputs, references)]
    return round(sum(scores) / len(scores), 4)

rouge_before = avg_rouge_l(before_outputs, REFERENCE_ANSWERS)
rouge_after  = avg_rouge_l(after_outputs,  REFERENCE_ANSWERS)
print(f'\n   ROUGE-L BEFORE: {rouge_before}')
print(f'   ROUGE-L AFTER:  {rouge_after}')

# Slang density (per 100 tokens)
def avg_slang_density(outputs):
    return round(sum(slang_density(o['text']) for o in outputs) / len(outputs), 2)

sd_before = avg_slang_density(before_outputs)
sd_after  = avg_slang_density(after_outputs)
print(f'\n   Slang density BEFORE: {sd_before}')
print(f'   Slang density AFTER:  {sd_after}')

# Avg response length
avg_len_before = round(sum(o['tokens'] for o in before_outputs) / len(before_outputs))
avg_len_after  = round(sum(o['tokens'] for o in after_outputs)  / len(after_outputs))
print(f'\n   Avg tokens BEFORE: {avg_len_before}')
print(f'   Avg tokens AFTER:  {avg_len_after}')

In [ ]:
# ── Cell K: Build results.json ────────────────────────────────────────────────
import datetime

comparisons = []
for i, (q, ref, b, a) in enumerate(zip(TEST_QUESTIONS, REFERENCE_ANSWERS, before_outputs, after_outputs)):
    comparisons.append({
        'id': i,
        'question': q,
        'reference_answer': ref,
        'before': {
            'output': b['text'],
            'tokens': b['tokens'],
            'slang_count': count_slang(b['text']),
        },
        'after': {
            'output': a['text'],
            'tokens': a['tokens'],
            'slang_count': count_slang(a['text']),
        },
    })

# Load dataset preview (first 10 training examples)
dataset_preview = examples[:10]

results = {
    'meta': {
        'generated_at': datetime.datetime.utcnow().isoformat() + 'Z',
        'base_model': MODEL_ID,
        'adapter_repo': 'your-username/qwen25-green-neko-genz',  # update after Hub push
        'training_samples': len(examples),
        'epochs': 3,
        'lora_rank': 16,
        'lora_alpha': 32,
        'is_placeholder': False,
    },
    'training_history': training_history,
    'metrics': {
        'perplexity_before': ppl_before,
        'perplexity_after':  ppl_after,
        'rouge_l_before':    rouge_before,
        'rouge_l_after':     rouge_after,
        'slang_density_before': sd_before,
        'slang_density_after':  sd_after,
        'avg_response_tokens_before': avg_len_before,
        'avg_response_tokens_after':  avg_len_after,
    },
    'comparisons': comparisons,
    'dataset_preview': dataset_preview,
}

with open('results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print('✅ results.json written')
print(json.dumps(results['metrics'], indent=2))

In [ ]:
# ── Cell L: Save to Google Drive + download ────────────────────────────────────
from google.colab import drive, files
import shutil

# Mount Drive
drive.mount('/content/drive')

# Save to Drive (prevents loss on session disconnect)
drive_dir = '/content/drive/MyDrive/fine-tuner-lab'
import os
os.makedirs(drive_dir, exist_ok=True)
shutil.copy('results.json', f'{drive_dir}/results.json')
print(f'✅ Saved to Google Drive: {drive_dir}/results.json')

# Also download directly
files.download('results.json')
print('✅ Download triggered — move results.json into results/ folder of the project')

In [ ]:
# ── Cell M: (Optional) Push adapter to HuggingFace Hub ────────────────────────
# Uncomment and fill in your HuggingFace token to push the adapter

# HF_TOKEN = 'hf_your_token_here'  # Get from https://huggingface.co/settings/tokens
# HF_REPO  = 'your-username/qwen25-green-neko-genz'
#
# from huggingface_hub import login
# login(token=HF_TOKEN)
#
# peft_model.push_to_hub(HF_REPO)
# tokenizer.push_to_hub(HF_REPO)
# print(f'✅ Adapter pushed to https://huggingface.co/{HF_REPO}')
# print('Update meta.adapter_repo in results.json with this URL')

print('ℹ️  Optional Hub push — uncomment lines above if you want to push the adapter')